In [1]:
# --- PROCESO CON REANUDACIÓN TOTAL Y PARALELISMO ---

import pandas as pd
import requests
import time
import ollama
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- CONFIGURACIÓN ---
MODELO = "gemma3:4b"  # Recomendado por velocidad
ARCHIVO_ENTRADA = "./datasets/dataset_steam.csv"
ARCHIVO_MAESTRO = "./datasets/juegos_maestro2.csv"


MAX_THREADS_STEAM = 3   # Máximo 2-3 para evitar baneos de Steam
MAX_THREADS_OLLAMA = 4  # Ajusta según tu GPU/RAM (4-8 suele ser ideal)
PAUSA_STEAM = 1.0       # Respiro extra entre peticiones concurrentes

if not os.path.exists('./datasets'): os.makedirs('./datasets')

# --- FUNCIONES DE APOYO ---

def obtener_info_juego(appid):
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&l=spanish"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 429:
            time.sleep(20)
            return None
        data = response.json()
        if data and data.get(str(appid), {}).get('success'):
            
            details = data[str(appid)]['data']
            genres_list = details.get('genres', [])
            date_str = details.get('release_date', {}).get('date', '')

            return {
                'appid': appid,
                'name': details.get('name', 'N/A'),
                'developer': details.get('developers', ["Desconocido"])[0],
                'genres_steam': ", ".join([g['description'] for g in genres_list]) if genres_list else "Sin Género",
                'price': 0 if details.get('is_free') else (details.get('price_overview', {}).get('final', 0) / 100),
                'country': None,
                'genre': None,
                'release_year': int(pd.to_datetime(date_str, errors='coerce').year),
                'metacritic_score': details.get('metacritic', {}).get('score', 0),
                'total_recommendations': details.get('recommendations', {}).get('total', 0),
                'is_free': details.get('is_free', False)
            }
    except: return None

def obtener_pais_worker(dev):
    """Worker para procesar países en paralelo."""
    prompt = f"""Empresa: {dev}. Eres un buscador de sedes corporativas.
    Tarea: Indica el país donde se encuentra la sede principal de la empresa: '{dev}'.
    
    Reglas críticas de respuesta:
    1. Responde ÚNICAMENTE con el nombre del país en español.
    2. NO escribas frases como "La sede está en..." o "El país es...".
    3. NO uses puntos finales ni signos de puntuación.
    4. NO des ninguna información adicional, ni ciudades, ni fechas.
    5. Si no conoces el país, responde exclusivamente con la palabra: Desconocido.
    6. Responde con exactamente UNA palabra.
    """
    try:
        response = ollama.chat(model=MODELO, messages=[{'role': 'user', 'content': prompt}])
        pais = response['message']['content'].strip().split('\n')[0].replace(".", "")
        return dev, pais
    except: return dev, "Error"

def obtener_genero_worker(combo_steam):
    """Clasifica un combo de etiquetas Steam en un género genérico."""
    generos_referencia = "Acción, RPG, Estrategia, Aventura, Simulación, Deportes, Puzzle, Shooter"
    
    # FIX: el parámetro es un combo de géneros Steam, no un nombre de juego
    prompt = f"""
    Eres un experto clasificador de videojuegos.
    Objetivo: A partir de estas etiquetas de Steam '{combo_steam}', determina el género principal más genérico.
    Reglas:
    1. Responde con exactamente UNA palabra.
    2. Género genérico, NO específico.
    3. Usa esta lista como preferencia: [{generos_referencia}].
    4. NO uses puntos ni explicaciones.
    """

    try:
        response = ollama.chat(model=MODELO, messages=[{'role': 'user', 'content': prompt}])
        genero = response['message']['content'].strip().split('\n')[0].replace(".", "").capitalize()
        return combo_steam, genero
    except Exception as e:
        return combo_steam, "Error"

# --- PROCESO PRINCIPAL ---

# 1. Cargar o Crear Dataset Maestro
if os.path.exists(ARCHIVO_MAESTRO):
    df_maestro = pd.read_csv(ARCHIVO_MAESTRO)
else:
    df_maestro = pd.DataFrame(columns=['appid', 'name', 'developer', 'genres_steam', 'price', 'country', 'genre', 'release_year', 'metacritic_score', 'total_recommendations', 'is_free' ])

# ─────────────────────────────────────────────────────────────
# PASO 1: Steam
# ─────────────────────────────────────────────────────────────
df_inicial = pd.read_csv(ARCHIVO_ENTRADA)
ids_totales = df_inicial["appid"].unique()
ids_faltantes = [aid for aid in ids_totales if aid not in df_maestro['appid'].values]

if ids_faltantes:
    print(f"--- PASO 1: Steam ({len(ids_faltantes)} juegos, {MAX_THREADS_STEAM} hilos) ---")
    with ThreadPoolExecutor(max_workers=MAX_THREADS_STEAM) as executor:
        futures = {executor.submit(obtener_info_juego, aid): aid for aid in ids_faltantes}
        nuevos_registros = []
        for f in tqdm(as_completed(futures), total=len(futures), desc="Steam"):
            res = f.result()
            if res:
                nuevos_registros.append(res)
            
            # Guardado incremental cada 20 resultados para no perder nada
            if len(nuevos_registros) >= 20:
                dfn = pd.DataFrame(nuevos_registros)
                if not dfn.empty:
                    # eliminar columnas totalmente vacías para evitar el FutureWarning
                    dfn = dfn.dropna(axis=1, how='all')
                    df_maestro = pd.concat([df_maestro, dfn], ignore_index=True)
                    df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)
                nuevos_registros = []
            time.sleep(PAUSA_STEAM / MAX_THREADS_STEAM)
        
        if nuevos_registros:
            dfn = pd.DataFrame(nuevos_registros)
            if not dfn.empty:
                dfn = dfn.dropna(axis=1, how='all')
                df_maestro = pd.concat([df_maestro, dfn], ignore_index=True)
                df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)

df_maestro.drop(columns=['country', 'genre'], inplace=True, errors='ignore')

# ─────────────────────────────────────────────────────────────
# PASO 2: Países
# ─────────────────────────────────────────────────────────────
print(f"\n--- PASO 2: Ollama Países ---")

devs_totales = df_maestro['developer'].unique()
print(f"Total de desarrolladores a procesar: {len(devs_totales)}")

with ThreadPoolExecutor(max_workers=MAX_THREADS_OLLAMA) as executor:
    futures = [executor.submit(obtener_pais_worker, d) for d in devs_totales]
    
    contador_pais = 0  # FIX: contador incremental real
    for f in tqdm(as_completed(futures), total=len(futures), desc="Países"):
        dev, pais = f.result()
        df_maestro.loc[df_maestro['developer'] == dev, 'country'] = pais
        
        contador_pais += 1
        if contador_pais % 20 == 0:  # FIX: ahora sí guarda cada 20 devs procesados
            df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)

# Guardado final Paso 2
df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)

# ─────────────────────────────────────────────────────────────
# PASO 3: Géneros
# ─────────────────────────────────────────────────────────────
print(f"\n--- PASO 3: Ollama Géneros ---")

combos_totales = df_maestro['genres_steam'].unique()
print(f"Total de combos únicos a reclasificar: {len(combos_totales)}")

with ThreadPoolExecutor(max_workers=MAX_THREADS_OLLAMA) as executor:
    futures = {executor.submit(obtener_genero_worker, c): c for c in combos_totales}
    
    contador_genero = 0
    for f in tqdm(as_completed(futures), total=len(futures), desc="Géneros"):
        combo_original = futures[f]
        try:
            _, genero_limpio = f.result()
            df_maestro.loc[df_maestro['genres_steam'] == combo_original, 'genre'] = genero_limpio
            
            contador_genero += 1
            if contador_genero % 15 == 0:
                df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)
        except Exception as e:
            print(f"Error en combo {combo_original}: {e}")

# Guardado final absoluto
df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)
print(f"\n✅ Proceso completo. Datos en {ARCHIVO_MAESTRO}")


--- PASO 1: Steam (12148 juegos, 3 hilos) ---


Steam: 100%|██████████| 12148/12148 [1:10:44<00:00,  2.86it/s]



--- PASO 2: Ollama Países ---
Total de desarrolladores a procesar: 420


Países: 100%|██████████| 420/420 [07:31<00:00,  1.07s/it]



--- PASO 3: Ollama Géneros ---
Total de combos únicos a reclasificar: 120


Géneros: 100%|██████████| 120/120 [01:30<00:00,  1.33it/s]


✅ Proceso completo. Datos en ./datasets/juegos_maestro2.csv
